# Day 22–26: Model Serving & SHAP Explainability
This Jupyter notebook demonstrates how to load the serialized assets (`model_assets.joblib`), make predictions on a new borrower input in Indian Rupees (INR), and explain the model's decision using SHAP waterfall plots.

In [ ]:
import joblib
import pandas as pd
import numpy as np
import shap
import os

# Define paths and load assets
assets_path = '../models/model_assets.joblib'
if not os.path.exists(assets_path):
    # Fallback for notebook running in workspace root
    assets_path = 'models/model_assets.joblib'

assets = joblib.load(assets_path)
model = assets['model']
features = assets['features']
medians = assets['medians']
winsorize_limits = assets['winsorize_limits']
cat_mappings = assets['cat_mappings']

print("Model assets loaded successfully!")
print(f"Number of input features expected by model: {len(features)}")

## Step 1: Simulate a New Borrower Profile
Let's define a new credit application in Indian Rupees (₹).

In [ ]:
# Currency Conversion Layer (1 USD = 83 INR)
CONVERSION_RATE = 83.0

# User inputs in INR
loan_amnt_inr = 1200000.0  # ₹12 Lakhs
annual_inc_inr = 1500000.0  # ₹15 Lakhs
installment_inr = 30000.0   # ₹30,000 monthly payment
revol_bal_inr = 450000.0    # ₹4.5 Lakhs revolving credit
int_rate = 14.5             # 14.5% interest rate
dti = 18.5                  # 18.5% Debt-to-Income ratio

# Start with medians to fill advanced features
sample_input = medians.copy()

# Insert core numerical inputs (converted to USD internally)
sample_input['loan_amnt'] = loan_amnt_inr / CONVERSION_RATE
sample_input['funded_amnt'] = loan_amnt_inr / CONVERSION_RATE
sample_input['funded_amnt_inv'] = loan_amnt_inr / CONVERSION_RATE
sample_input['int_rate'] = float(int_rate)
sample_input['installment'] = installment_inr / CONVERSION_RATE
sample_input['annual_inc'] = annual_inc_inr / CONVERSION_RATE
sample_input['dti'] = float(dti)
sample_input['revol_bal'] = revol_bal_inr / CONVERSION_RATE
sample_input['income_to_loan_ratio'] = annual_inc_inr / (loan_amnt_inr + 1e-5)

# Preprocess categorical features (term, grade, etc.)
categorical_inputs = {
    'term': ' 36 months',
    'grade': 'B',
    'sub_grade': 'B4',
    'emp_length': '10+ years',
    'home_ownership': 'MORTGAGE',
    'verification_status': 'Verified',
    'purpose': 'debt_consolidation',
    'application_type': 'Individual',
    'disbursement_method': 'Cash',
    'initial_list_status': 'w',
    'title': 'Debt consolidation'
}

for col, val in categorical_inputs.items():
    if col in cat_mappings:
        freq_map = cat_mappings[col]['freq_map']
        default_mode = cat_mappings[col]['default_mode']
        val_str = str(val)
        sample_input[col] = freq_map[val_str] if val_str in freq_map else freq_map[default_mode]

# Clip numerical outliers (Winsorization matching train set)
for col in winsorize_limits:
    if col in sample_input:
        limits = winsorize_limits[col]
        sample_input[col] = np.clip(sample_input[col], limits['lower'], limits['upper'])

# Convert to DataFrame and align column orders
input_df = pd.DataFrame([sample_input])[features]
print("Preprocessed borrower feature vector:")
input_df

## Step 2: Scoring Probability of Default & Expected Loss

In [ ]:
pd_prob = model.predict_proba(input_df)[0, 1]
lgd = 0.50  # Loss Given Default (50% standard rate)
expected_loss_inr = pd_prob * lgd * loan_amnt_inr

print(f"Predicted Probability of Default (PD): {pd_prob * 100:.2f}%")
print(f"Expected Loss (EL) in INR: ₹{expected_loss_inr:,.2f}")

## Step 3: Local Decision Interpretability (SHAP Waterfall)
Let's explain the decision locally to show exactly which inputs increased or decreased the borrower's default risk.

In [ ]:
explainer = shap.TreeExplainer(model)
shap_values = explainer(input_df)

# Plot the SHAP waterfall for this specific loan application
shap.plots.waterfall(shap_values[0])